In [ ]:
#block 1 - imports
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from scipy.optimize import minimize

In [ ]:
#block 2 - loading data
df = pd.read_csv('imi_meteorites_combined.csv')

feature_cols = [c for c in df.columns if c not in ['Sample Name', 'Sample SubType', 'Class']]
X = df[feature_cols].values
y = df['Class'].values

print("Dataset shape:", df.shape)
print("Classes:", list(df['Class'].unique()))
print("Features:", feature_cols)

Dataset shape: (245, 67)
Classes: ['H', 'Enstatite', 'L', 'Carbonaceous', 'Achondrite', 'LL']
Features: ['SiO2 (wt%)', 'TiO2 (vol%)', 'Al2O3 (wt%)', 'FeO (vol%)', 'MgO (wt%)', 'MnO (wt%)', 'CaO (wt%)', 'Na2O (wt%)', 'K2O (wt%)', 'P2O5 (wt%)', 'Cr2O3 (vol%)', 'Fe (wt%)', 'Cr (wt%)', 'Mn (wt%)', 'Mg (wt%)', 'Ca (wt%)', 'Al (wt%)', 'Na (wt%)', 'Ti (wt%)', 'La (ppm)', 'Sm (ppm)', 'Eu (ppm)', 'Yb (ppm)', 'Lu (ppm)', 'Ba (ppm)', 'He3 (ccstp/g)', 'He4 (ccstp/g)', 'Ne20 (ccstp/g)', 'Ne21 (ccstp/g)', 'Ne22 (ccstp/g)', 'Ar36 (ccstp/g)', 'Ar38 (ccstp/g)', 'Ar40 (ccstp/g)', 'FeTOT (wt%)', 'FeS (wt%)', 'Fe/Mg', 'MnO/MgO', 'FeO/MgO', 'Al2O3/CaO', 'La/Lu', 'Eu/Sm', 'He3/He4', 'Ne21/Ne22', 'Ar40/Ar36', 'Total REE', 'major_oxides_mean', 'iron_related_mean', 'ree_mean', 'he_group_mean', 'ne_group_mean', 'ar_group_mean', 'lithophile_mean', 'siderophile_mean', 'all_noble_gases_mean', 'elemental_weight_mean', 'Fe_Oxidation_Index', 'Mg_Si_Ratio', 'Mafic_Index', 'Eu_Anomaly', 'Silica_Saturation_Index', 'Mg_N

In [ ]:
#block 3 - encoding labels and scaling features
le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Training samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])

Training samples: 196
Test samples: 49


In [ ]:
#block 4 - training forward model
clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
clf.fit(X_train_scaled, y_train)
print("Forward model trained successfully")

Forward model trained successfully


In [ ]:
#block 4.5 - exporting RF + scaler + metadata to ONNX
!pip install -q skl2onnx onnx

from skl2onnx import to_onnx
from skl2onnx.common.data_types import FloatTensorType
import numpy as np, json

n_features = X_train_scaled.shape[1]
initial_type = [("input", FloatTensorType([None, n_features]))]

#export the classifier (operates on scaled inputs)
onnx_clf = to_onnx(
    clf,
    initial_types=initial_type,
    target_opset=15,
    options={id(clf): {"zipmap": False}},  # outputs raw probability array
)
with open("meteorite_rf.onnx", "wb") as f:
    f.write(onnx_clf.SerializeToString())

#export scaler params + class order + per-class training means (for inverse design init)
metadata = {
    "feature_cols": feature_cols,
    "classes": le.classes_.tolist(),
    "scaler_mean": scaler.mean_.tolist(),
    "scaler_scale": scaler.scale_.tolist(),
    "feature_min": X.min(axis=0).tolist(),
    "feature_max": X.max(axis=0).tolist(),
    "class_means": {
        cls: X_train[y_train == le.transform([cls])[0]].mean(axis=0).tolist()
        for cls in le.classes_
    },
}
with open("meteorite_meta.json", "w") as f:
    json.dump(metadata, f)

from google.colab import files
files.download("meteorite_rf.onnx")
files.download("meteorite_meta.json")
print("Done — upload both files to the website chat.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 15.7 MB/s eta 0:00:00


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done — upload both files to the website chat.


In [ ]:
#block 5 - evaluating forward model
y_pred = clf.predict(X_test_scaled)
print("=== Forward Model Performance ===")
print(classification_report(y_test, y_pred, target_names=le.classes_))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

=== Forward Model Performance ===
              precision    recall  f1-score   support

  Achondrite       1.00      1.00      1.00        10
Carbonaceous       1.00      1.00      1.00         7
   Enstatite       1.00      1.00      1.00         4
           H       0.90      0.90      0.90        10
           L       0.90      0.90      0.90        10
          LL       1.00      1.00      1.00         8

    accuracy                           0.96        49
   macro avg       0.97      0.97      0.97        49
weighted avg       0.96      0.96      0.96        49

Confusion Matrix:
[[10  0  0  0  0  0]
 [ 0  7  0  0  0  0]
 [ 0  0  4  0  0  0]
 [ 0  0  0  9  1  0]
 [ 0  0  0  1  9  0]
 [ 0  0  0  0  0  8]]


In [ ]:
#block 6 - inverse design function
def inverse_design(target_class, features_to_optimize):
    target_idx = le.transform([target_class])[0]
    opt_indices = [feature_cols.index(f) for f in features_to_optimize]

    class_mask = y_train == target_idx
    class_mean = X_train[class_mask].mean(axis=0)
    x0 = class_mean[opt_indices]

    bounds = [(X[:, i].min(), X[:, i].max()) for i in opt_indices]

    def objective(x_opt):
        x_full = class_mean.copy()
        for idx, val in zip(opt_indices, x_opt):
            x_full[idx] = val
        x_scaled = scaler.transform(x_full.reshape(1, -1))
        proba = clf.predict_proba(x_scaled)[0][target_idx]
        return -proba

    result = minimize(objective, x0, method='L-BFGS-B', bounds=bounds)

    print(f"\n=== Inverse Design Result ===")
    print(f"Target Class : {target_class}")
    print(f"Optimized Features:")
    for fname, val in zip(features_to_optimize, result.x):
        print(f"  {fname}: {val:.6f}")
    print(f"Confidence   : {-result.fun:.4f}")
    return dict(zip(features_to_optimize, result.x))

In [ ]:
#block 7 - running inverse design
target_class = input("Enter target class (H / L / LL / Carbonaceous / Achondrite / Enstatite): ")
n_features = 5

features_to_optimize = []
print(f"\nAvailable features:\n{feature_cols}")
for i in range(n_features):
    f = input(f"Enter feature {i+1} name exactly as shown above: ")
    features_to_optimize.append(f)

result = inverse_design(target_class, features_to_optimize)

Enter target class (H / L / LL / Carbonaceous / Achondrite / Enstatite): H

Available features:
['SiO2 (wt%)', 'TiO2 (vol%)', 'Al2O3 (wt%)', 'FeO (vol%)', 'MgO (wt%)', 'MnO (wt%)', 'CaO (wt%)', 'Na2O (wt%)', 'K2O (wt%)', 'P2O5 (wt%)', 'Cr2O3 (vol%)', 'Fe (wt%)', 'Cr (wt%)', 'Mn (wt%)', 'Mg (wt%)', 'Ca (wt%)', 'Al (wt%)', 'Na (wt%)', 'Ti (wt%)', 'La (ppm)', 'Sm (ppm)', 'Eu (ppm)', 'Yb (ppm)', 'Lu (ppm)', 'Ba (ppm)', 'He3 (ccstp/g)', 'He4 (ccstp/g)', 'Ne20 (ccstp/g)', 'Ne21 (ccstp/g)', 'Ne22 (ccstp/g)', 'Ar36 (ccstp/g)', 'Ar38 (ccstp/g)', 'Ar40 (ccstp/g)', 'FeTOT (wt%)', 'FeS (wt%)', 'Fe/Mg', 'MnO/MgO', 'FeO/MgO', 'Al2O3/CaO', 'La/Lu', 'Eu/Sm', 'He3/He4', 'Ne21/Ne22', 'Ar40/Ar36', 'Total REE', 'major_oxides_mean', 'iron_related_mean', 'ree_mean', 'he_group_mean', 'ne_group_mean', 'ar_group_mean', 'lithophile_mean', 'siderophile_mean', 'all_noble_gases_mean', 'elemental_weight_mean', 'Fe_Oxidation_Index', 'Mg_Si_Ratio', 'Mafic_Index', 'Eu_Anomaly', 'Silica_Saturation_Index', 'Mg_Number', 

In [ ]:
#block 8 - forward model user input

fundamental_features = [c for c in feature_cols if c in [
    'SiO2 (wt%)', 'TiO2 (vol%)', 'Al2O3 (wt%)', 'FeO (vol%)', 'MgO (wt%)',
    'MnO (wt%)', 'CaO (wt%)', 'Na2O (wt%)', 'K2O (wt%)', 'P2O5 (wt%)',
    'Cr2O3 (vol%)', 'Fe (wt%)', 'Cr (wt%)', 'Mn (wt%)', 'Mg (wt%)',
    'Ca (wt%)', 'Al (wt%)', 'Na (wt%)', 'Ti (wt%)', 'La (ppm)',
    'Sm (ppm)', 'Eu (ppm)', 'Yb (ppm)', 'Lu (ppm)', 'Ba (ppm)',
    'He3 (ccstp/g)', 'He4 (ccstp/g)', 'Ne20 (ccstp/g)', 'Ne21 (ccstp/g)',
    'Ne22 (ccstp/g)', 'Ar36 (ccstp/g)', 'Ar38 (ccstp/g)', 'Ar40 (ccstp/g)',
    'FeTOT (wt%)', 'FeS (wt%)'
]]

def forward_model_predict(features_to_input):
    df_features = pd.DataFrame(X, columns=feature_cols)

    print("\nEnter values for your features:")
    user_values = {}
    valid = True

    for fname in features_to_input:
        if fname not in fundamental_features:
            print(f"  WARNING: {fname} is a derived feature and cannot be directly measured in a lab.")
            valid = False
            continue
        col_min = df_features[fname].min()
        col_max = df_features[fname].max()
        print(f"  {fname} — valid range: [{col_min:.4f}, {col_max:.4f}]")
        val = float(input(f"  Enter value for {fname}: "))

        if val < col_min or val > col_max:
            print(f"  WARNING: Value outside realistic range — flagged as invalid")
            valid = False
        user_values[fname] = val

    if not valid:
        print("\nInvalid input — please re-enter with valid fundamental features and realistic values.")
        return

    # Step 1 - Fill all features with overall mean, substitute user inputs
    overall_mean = X.mean(axis=0)
    x_full = overall_mean.copy()
    for fname, val in user_values.items():
        idx = feature_cols.index(fname)
        x_full[idx] = val

    # Step 2 - Rough prediction
    x_scaled = scaler.transform(x_full.reshape(1, -1))
    rough_class_idx = clf.predict(x_scaled)[0]
    rough_class = le.inverse_transform([rough_class_idx])[0]

    # Step 3 - Fill remaining features with predicted class mean
    class_mask = y == rough_class
    class_mean = X[class_mask].mean(axis=0)
    x_refined = class_mean.copy()
    for fname, val in user_values.items():
        idx = feature_cols.index(fname)
        x_refined[idx] = val

    # Step 4 - Final prediction
    x_refined_scaled = scaler.transform(x_refined.reshape(1, -1))
    final_proba = clf.predict_proba(x_refined_scaled)[0]
    final_class_idx = final_proba.argmax()
    final_class = le.inverse_transform([final_class_idx])[0]
    confidence = final_proba[final_class_idx]

    print(f"\n=== Forward Model Result ===")
    print(f"Predicted Class : {final_class}")
    print(f"Confidence      : {confidence:.4f}")
    print(f"\nClass probabilities:")
    for cls, prob in zip(le.classes_, final_proba):
        print(f"  {cls}: {prob:.4f}")

# Run forward model
print(f"Available fundamental features ({len(fundamental_features)} total, directly measurable in a lab):")
print(fundamental_features)

n = int(input(f"\nHow many features do you want to input? (min 3, max {len(fundamental_features)}): "))
if n < 3:
    print("Minimum is 3 features. Setting to 3.")
    n = 3
if n > len(fundamental_features):
    print(f"Maximum is {len(fundamental_features)} features. Setting to {len(fundamental_features)}.")
    n = len(fundamental_features)

features_to_input = []
for i in range(n):
    f = input(f"Enter feature {i+1} to input: ")
    features_to_input.append(f)

forward_model_predict(features_to_input)

Available features:
['SiO2 (wt%)', 'TiO2 (vol%)', 'Al2O3 (wt%)', 'FeO (vol%)', 'MgO (wt%)', 'MnO (wt%)', 'CaO (wt%)', 'Na2O (wt%)', 'K2O (wt%)', 'P2O5 (wt%)', 'Cr2O3 (vol%)', 'Fe (wt%)', 'Cr (wt%)', 'Mn (wt%)', 'Mg (wt%)', 'Ca (wt%)', 'Al (wt%)', 'Na (wt%)', 'Ti (wt%)', 'La (ppm)', 'Sm (ppm)', 'Eu (ppm)', 'Yb (ppm)', 'Lu (ppm)', 'Ba (ppm)', 'He3 (ccstp/g)', 'He4 (ccstp/g)', 'Ne20 (ccstp/g)', 'Ne21 (ccstp/g)', 'Ne22 (ccstp/g)', 'Ar36 (ccstp/g)', 'Ar38 (ccstp/g)', 'Ar40 (ccstp/g)', 'FeTOT (wt%)', 'FeS (wt%)', 'Fe/Mg', 'MnO/MgO', 'FeO/MgO', 'Al2O3/CaO', 'La/Lu', 'Eu/Sm', 'He3/He4', 'Ne21/Ne22', 'Ar40/Ar36', 'Total REE', 'major_oxides_mean', 'iron_related_mean', 'ree_mean', 'he_group_mean', 'ne_group_mean', 'ar_group_mean', 'lithophile_mean', 'siderophile_mean', 'all_noble_gases_mean', 'elemental_weight_mean', 'Fe_Oxidation_Index', 'Mg_Si_Ratio', 'Mafic_Index', 'Eu_Anomaly', 'Silica_Saturation_Index', 'Mg_Number', 'Volatile_Depletion_Index', 'Refractory_Volatile_Ratio', 'Metal_Silicate_Pro